# B2B Customer Churn — XGBoost Model Training & Explainability

**Purpose:** Train a production-grade binary churn classifier on preprocessed features, evaluate business-relevant metrics, and explain predictions with SHAP.

**Prerequisites:** Run `feature_engineering.ipynb` to generate artifacts in `models/artifacts/`.

**Outputs:**
- `models/churn_xgboost_model.joblib` — trained estimator
- Evaluation plots and risk-scored prediction table (in-notebook)


In [ ]:
# --- Imports ---
from pathlib import Path
import warnings

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
import xgboost as xgb
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.figsize": (10, 6), "figure.dpi": 100})

RANDOM_STATE = 42
ARTIFACTS_DIR = Path("../models/artifacts")
MODEL_PATH = Path("../models/churn_xgboost_model.joblib")

# Risk tier thresholds (tune with CS ops capacity)
RISK_THRESHOLDS = {"low": 0.33, "high": 0.66}


## 1. Load preprocessed artifacts

Features were scaled/encoded in the feature engineering pipeline. We train on the **same transformed matrices** to stay consistent with inference.


In [ ]:
X_train = joblib.load(ARTIFACTS_DIR / "X_train.joblib")
X_test = joblib.load(ARTIFACTS_DIR / "X_test.joblib")
y_train = joblib.load(ARTIFACTS_DIR / "y_train.joblib")
y_test = joblib.load(ARTIFACTS_DIR / "y_test.joblib")
feature_names = joblib.load(ARTIFACTS_DIR / "feature_names.joblib")

# Ensure numpy arrays for XGBoost / SHAP
if hasattr(y_train, "values"):
    y_train = y_train.values
if hasattr(y_test, "values"):
    y_test = y_test.values

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Features:", len(feature_names))
print(f"Train churn rate: {y_train.mean():.2%}")
print(f"Test churn rate:  {y_test.mean():.2%}")


## 2. Class imbalance strategy

B2B churn is typically **imbalanced** (more retained than churned accounts). We use XGBoost `scale_pos_weight` — the ratio of negative to positive class counts — so the model penalizes missed churners appropriately without oversampling artifacts.


In [ ]:
neg_count = (y_train == 0).sum()
pos_count = (y_train == 1).sum()
scale_pos_weight = neg_count / pos_count

print(f"Negatives (retained): {neg_count:,}")
print(f"Positives (churned):  {pos_count:,}")
print(f"scale_pos_weight:     {scale_pos_weight:.2f}")


## 3. XGBoost baseline configuration

Strong default hyperparameters for tabular churn: moderate depth, subsampling for generalization, and `logloss` for probabilistic ranking quality.


In [ ]:
xgb_model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=0.1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    scale_pos_weight=scale_pos_weight,
    eval_metric="logloss",
    objective="binary:logistic",
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method="hist",
)

xgb_model.fit(
    X_train,
    y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=False,
)

print("Training complete.")


## 4. Test-set evaluation

For churn, **recall** (catching true churners) and **ROC-AUC / PR-AUC** (ranking quality) often matter more than raw accuracy under imbalance.


In [ ]:
y_pred = xgb_model.predict(X_test)
y_proba = xgb_model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision": precision_score(y_test, y_pred, zero_division=0),
    "recall": recall_score(y_test, y_pred, zero_division=0),
    "f1_score": f1_score(y_test, y_pred, zero_division=0),
    "roc_auc": roc_auc_score(y_test, y_proba),
}

metrics_df = pd.DataFrame([metrics]).T.round(4)
metrics_df.columns = ["score"]
metrics_df.index.name = "metric"
metrics_df


In [ ]:
print(classification_report(y_test, y_pred, target_names=["Retained", "Churned"]))


### Business interpretation — performance

- **Accuracy** can look acceptable even when churn recall is poor — do not use it alone for success criteria.
- **Precision** reflects how many flagged accounts truly churn — low precision wastes CSM outreach capacity.
- **Recall** reflects coverage of actual churners — low recall means silent revenue leakage.
- **F1** balances precision/recall when outreach cost and miss cost are similar.
- **ROC-AUC** measures ranking ability across thresholds — useful for prioritizing a fixed outreach list.

**False positives (predict churn, stay):** Wasted saves motion; acceptable if playbooks are low-cost.
**False negatives (predict retain, churn):** Highest business cost — monitor recall and consider lowering the probability threshold for enterprise segments.


## 5. Evaluation visualizations


In [ ]:
# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["Pred Retained", "Pred Churned"],
    yticklabels=["Actual Retained", "Actual Churned"],
    ax=ax,
)
ax.set_title("Confusion Matrix (Test Set)")
ax.set_ylabel("Actual")
ax.set_xlabel("Predicted")
plt.tight_layout()
plt.show()


In [ ]:
# ROC curve
fpr, tpr, _ = roc_curve(y_test, y_proba)
fig, ax = plt.subplots()
ax.plot(fpr, tpr, label=f"ROC-AUC = {metrics['roc_auc']:.3f}")
ax.plot([0, 1], [0, 1], "k--", label="Random")
ax.set_xlabel("False Positive Rate")
ax.set_ylabel("True Positive Rate (Recall)")
ax.set_title("ROC Curve")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Precision-Recall curve
prec, rec, _ = precision_recall_curve(y_test, y_proba)
fig, ax = plt.subplots()
ax.plot(rec, prec, label="PR Curve")
ax.axhline(y_test.mean(), color="k", linestyle="--", label=f"Baseline prevalence ({y_test.mean():.1%})")
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("Precision-Recall Curve")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# XGBoost feature importance (gain)
importance = pd.Series(
    xgb_model.feature_importances_,
    index=feature_names,
    name="importance",
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
importance.head(20).sort_values().plot(kind="barh", ax=ax)
ax.set_title("Top 20 Features — XGBoost Gain Importance")
ax.set_xlabel("Importance (gain)")
plt.tight_layout()
plt.show()

print("Top churn-driving features (gain):")
print(importance.head(10).to_string())


### Churn driver analysis (model-based)

Gain importance highlights which features the trees split on most often. Combine with SHAP (next section) for **directional** effects (increases vs decreases churn risk).


## 6. Churn probabilities & risk tiers

`predict_proba` outputs calibrated-ish risk scores for prioritization. Risk bands translate scores into CS playbook tiers.


In [ ]:
def assign_risk_category(proba: float) -> str:
    if proba < RISK_THRESHOLDS["low"]:
        return "Low Risk"
    if proba < RISK_THRESHOLDS["high"]:
        return "Medium Risk"
    return "High Risk"


predictions_df = pd.DataFrame({
    "churn_probability": y_proba,
    "predicted_churn": y_pred,
    "actual_churn": y_test,
    "risk_category": [assign_risk_category(p) for p in y_proba],
})

predictions_df.head(10)


In [ ]:
# Risk tier distribution on test set
predictions_df["risk_category"].value_counts().sort_index()


**Tier guidance:**
- **Low Risk:** Monitor only; automate health checks.
- **Medium Risk:** Proactive QBR / usage review.
- **High Risk:** Executive outreach, discount/renewal save plays, support escalation audit.


## 7. Persist trained model


In [ ]:
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(xgb_model, MODEL_PATH)
print(f"Model saved to {MODEL_PATH.resolve()}")


## 8. SHAP explainability

SHAP (SHapley Additive exPlanations) decomposes each prediction into feature contributions — critical for B2B stakeholders who need **why** an account is at risk.

We use `TreeExplainer` for XGBoost. Summary plots use a sample of the test set for speed.


In [ ]:
# SHAP explainer (TreeExplainer for boosted trees)
explainer = shap.TreeExplainer(xgb_model)

# Sample for global plots (performance)
SHAP_SAMPLE_SIZE = min(500, X_test.shape[0])
rng = np.random.default_rng(RANDOM_STATE)
shap_idx = rng.choice(X_test.shape[0], size=SHAP_SAMPLE_SIZE, replace=False)
X_shap = X_test[shap_idx]

def to_shap_matrix(values):
    """Normalize SHAP outputs to (n_samples, n_features) for binary classification."""
    if isinstance(values, list):
        values = values[1] if len(values) > 1 else values[0]
    arr = np.asarray(values)
    if arr.ndim == 3:
        arr = arr[:, :, 1]
    return arr


shap_values = to_shap_matrix(explainer.shap_values(X_shap))
print("SHAP values shape:", shap_values.shape)


In [ ]:
# SHAP summary plot (beeswarm)
shap.summary_plot(
    shap_values,
    X_shap,
    feature_names=feature_names,
    show=False,
)
plt.title("SHAP Summary — Impact on Churn Risk")
plt.tight_layout()
plt.show()


In [ ]:
# SHAP mean absolute importance (global)
shap_importance = pd.Series(
    np.abs(shap_values).mean(axis=0),
    index=feature_names,
    name="mean_abs_shap",
).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 8))
shap_importance.head(20).sort_values().plot(kind="barh", ax=ax, color=sns.color_palette()[1])
ax.set_title("Top 20 Features — Mean |SHAP|")
ax.set_xlabel("Mean |SHAP value|")
plt.tight_layout()
plt.show()

print("Top churn-driving features (SHAP):")
print(shap_importance.head(10).to_string())


In [ ]:
# SHAP waterfall — single high-risk example from test set
example_idx = int(np.argmax(y_proba))
example_x = X_test[example_idx : example_idx + 1]
example_shap = to_shap_matrix(explainer.shap_values(example_x))[0]

base_value = explainer.expected_value
if isinstance(base_value, (list, np.ndarray)):
    flat = np.array(base_value).flatten()
    base_value = float(flat[1]) if len(flat) > 1 else float(flat[0])
else:
    base_value = float(base_value)

shap.waterfall_plot(
    shap.Explanation(
        values=example_shap,
        base_values=base_value,
        data=example_x[0],
        feature_names=list(feature_names),
    ),
    show=False,
)
plt.title(
    f"SHAP Waterfall — Example Customer (P(churn)={y_proba[example_idx]:.2f}, "
    f"Tier={assign_risk_category(y_proba[example_idx])})"
)
plt.tight_layout()
plt.show()

print(f"Example index: {example_idx}")
print(f"Actual churn: {y_test[example_idx]} | Predicted: {y_pred[example_idx]}")


### Business interpretation — SHAP

- **Summary plot:** Each point is a customer; color is feature value; horizontal position is impact on churn log-odds. Red/high values pushing right increase risk for that feature.
- **Mean |SHAP|:** Global driver ranking — align save plays with the top 5–10 signals (e.g., satisfaction risk, usage, support friction).
- **Waterfall:** Narrative for a single account — use in QBRs or success escalations to explain *why* this logo is in the High Risk queue.

Reconcile XGBoost gain vs SHAP: gain shows split frequency; SHAP shows marginal contribution direction per prediction.


## 9. Summary & deployment checklist

| Item | Location |
|------|----------|
| Preprocessor | `models/artifacts/churn_preprocessor.joblib` |
| Trained model | `models/churn_xgboost_model.joblib` |
| Feature names | `models/artifacts/feature_names.joblib` |

**Next steps:** FastAPI scoring endpoint, Dash dashboard, threshold tuning with finance (cost of outreach vs LTV saved), and periodic retraining on fresh cohorts.
